In [2]:
import pandas as pd

df = pd.read_csv("../data/results.csv")

print(df.head())
print(df.shape)
print(df.columns)

         date home_team away_team  home_score  away_score tournament     city  \
0  1872-11-30  Scotland   England         0.0         0.0   Friendly  Glasgow   
1  1873-03-08   England  Scotland         4.0         2.0   Friendly   London   
2  1874-03-07  Scotland   England         2.0         1.0   Friendly  Glasgow   
3  1875-03-06   England  Scotland         2.0         2.0   Friendly   London   
4  1876-03-04  Scotland   England         3.0         0.0   Friendly  Glasgow   

    country  neutral  
0  Scotland    False  
1   England    False  
2  Scotland    False  
3   England    False  
4  Scotland    False  
(49287, 9)
Index(['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'tournament', 'city', 'country', 'neutral'],
      dtype='object')


In [9]:
home_df = pd.DataFrame({
    "date": df["date"],
    "team": df["home_team"],
    "opponent": df["away_team"],
    "goals_for": df["home_score"],
    "goals_against": df["away_score"],
    "tournament": df["tournament"],
    "neutral": df["neutral"]
})

away_df = pd.DataFrame({
    "date": df["date"],
    "team": df["away_team"],
    "opponent": df["home_team"],
    "goals_for": df["away_score"],
    "goals_against": df["home_score"],
    "tournament": df["tournament"],
    "neutral": df["neutral"]
})

team_matches = pd.concat([home_df, away_df], ignore_index=True)
team_matches = team_matches.sort_values("date").reset_index(drop=True)

def get_team_result(row):
    if row["goals_for"] > row["goals_against"]:
        return "Win"
    elif row["goals_for"] < row["goals_against"]:
        return "Loss"
    return "Draw"

team_matches["result"] = team_matches.apply(get_team_result, axis=1)
team_matches["points"] = team_matches["result"].map({
    "Win": 3,
    "Draw": 1,
    "Loss": 0
})
team_matches["goal_diff"] = team_matches["goals_for"] - team_matches["goals_against"]

In [7]:
def get_result(row):
    if row["home_score"] > row["away_score"]:
        return "HomeWin"
    elif row["home_score"] < row["away_score"]:
        return "AwayWin"
    else:
        return "Draw"

df["result"] = df.apply(get_result, axis=1)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

df[["home_team", "away_team", "home_score", "away_score", "result"]].head()

,home_team,away_team,home_score,away_score,result
0,Scotland,England,0.0,0.0,Draw
1,England,Scotland,4.0,2.0,HomeWin
2,Scotland,England,2.0,1.0,HomeWin
3,England,Scotland,2.0,2.0,Draw
4,Scotland,England,3.0,0.0,HomeWin


In [8]:
tournament_weights = {
    "FIFA World Cup": 1.5,
    "UEFA Euro": 1.3,
    "Copa América": 1.3,
    "AFC Asian Cup": 1.2,
    "African Cup of Nations": 1.2,
    "FIFA World Cup qualification": 1.1,
    "UEFA Nations League": 1.0,
    "Friendly": 0.6
}

match_weight = recency_weight * tournament_weight
recent_tournament_score
strength_score = (
    0.6 * form_score +
    0.25 * goal_diff_score +
    0.15 * tournament_success_score
)


NameError: name 'team_matches' is not defined

In [4]:
X = df[["home_team", "away_team", "tournament","neutral"]].copy()
y = df["result"].copy()

print(X.head())
print(y.head())

  home_team away_team tournament  neutral
0  Scotland   England   Friendly    False
1   England  Scotland   Friendly    False
2  Scotland   England   Friendly    False
3   England  Scotland   Friendly    False
4  Scotland   England   Friendly    False
0       Draw
1    HomeWin
2    HomeWin
3       Draw
4    HomeWin
Name: result, dtype: object


In [5]:
from sklearn.preprocessing import LabelEncoder

enc_home = LabelEncoder()
enc_away = LabelEncoder()
enc_tournament = LabelEncoder()
enc_result = LabelEncoder()

X["home_team"] = enc_home.fit_transform(X["home_team"])
X["away_team"] = enc_away.fit_transform(X["away_team"])
X["tournament"] = enc_tournament.fit_transform(X["tournament"])

y_encoded = enc_result.fit_transform(y)

print(X.head())
print(y_encoded[:5])
print(enc_result.classes_)

   home_team  away_team  tournament  neutral
0        248         87          91    False
1         86        242          91    False
2        248         87          91    False
3         86        242          91    False
4        248         87          91    False
[1 2 2 1 2]
['AwayWin' 'Draw' 'HomeWin']


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=enc_result.classes_))

Accuracy: 0.48681274092107935
              precision    recall  f1-score   support

     AwayWin       0.44      0.41      0.42      2821
        Draw       0.26      0.21      0.23      2227
     HomeWin       0.59      0.66      0.62      4810

    accuracy                           0.49      9858
   macro avg       0.43      0.43      0.43      9858
weighted avg       0.47      0.49      0.48      9858

